# NucleusRV: An Open-Source RV32IMAFC RISC-V Core
### A Technical Walkthrough

**Project of:** Microelectronics Research Lab (MERL), UIT University

**Repository:** [github.com/merledu/nucleusrv](https://github.com/merledu/nucleusrv)

**License:** GPL-3.0

**Submitted for:** IEEE SSCS Open-Source Ecosystem Code-a-Chip Travel Grant, ISSCC 2027

This notebook is a walkthrough of NucleusRV, an open-source 32-bit RISC-V processor core. It is a written explanation rather than a live simulation. It covers the core's architecture, the instruction set it supports, the pipeline design, how hazards are handled, how the core is verified, and its silicon fabrication history. A reader with no prior exposure to the project should be able to follow how it works and why it is useful for the open-source hardware community.

## 1. What is NucleusRV?

NucleusRV is a 32-bit, 5-stage pipelined RISC-V CPU core written in Chisel, a hardware construction language built on top of Scala. Chisel designs compile down to synthesizable Verilog, so NucleusRV can be used in an FPGA project or taken through to an ASIC (silicon chip) design flow.

The core implements the RV32IMAFC instruction set:

| Letter | Extension | What it adds |
|---|---|---|
| I | Base Integer ISA | The mandatory core instruction set: arithmetic, logic, loads and stores, branches |
| M | Multiply/Divide | Hardware integer multiplication and division |
| A | Atomics | Read-modify-write memory operations used for safe multi-threaded code |
| F | Single-Precision Floating Point | Hardware float arithmetic, built on the open-source berkeley-hardfloat library |
| C | Compressed Instructions | 16-bit versions of common instructions that shrink program size |

Every extension besides the mandatory base I set can be turned on or off at build time, along with the Zicsr extension (Control and Status Register instructions used for interrupts, counters, and privileged state). This makes NucleusRV a configurable core for teaching and research rather than a fixed design.

## 2. Why This Matters for Open-Source Silicon

A lot of chip design still happens with proprietary tools and closed source. NucleusRV is one of a growing set of processor cores that are:

- Fully open-source (GPL-3.0 licensed, with source and testbenches public on GitHub)
- Built with open-source tooling: Chisel and Scala for design, Verilator for simulation, and the RISC-V GNU toolchain for compiling test programs
- Verified against the official RISC-V architectural compliance suite, not just informal spot-checks

NucleusRV has also been taped out in silicon. An earlier RV32IMC configuration of the core was fabricated on Google's sponsored OpenMPW-6 shuttle using the SkyWater SKY130nm open-source PDK (Process Design Kit). This is not only a paper design. It is a processor that has been manufactured as a real chip using open-source PDK infrastructure, which is the kind of open-source work this grant program supports.

## 3. Design Decisions

A few design choices shape how NucleusRV is built, and they are worth calling out directly.

**Chisel instead of hand-written Verilog.** Writing the core in Chisel lets the same source describe many configurations. Enabling or disabling the M, A, F, and C extensions is a build-time parameter rather than a separate copy of the design. This keeps one codebase instead of several.

**A classic 5-stage pipeline.** The design uses the textbook IF, ID, EX, MEM, WB structure. This is a deliberate choice for a teaching and research core. The theory from a computer architecture course maps directly onto the real code, so the project is readable by students, while still being a working core and not a toy.

**Early branch resolution in the decode stage.** Branches and jumps are resolved in the ID stage rather than later. This limits the penalty of a taken branch to a single cycle, which keeps the simple pipeline close to one instruction per cycle without needing a complex branch predictor.

**Separate instruction and data memory ports (Harvard style).** Keeping the two memory ports independent lets the core fetch the next instruction in the same cycle it reads or writes data memory. This is part of what lets a simple design sustain high throughput.

**Configurable extensions over a fixed ISA.** Rather than committing to one instruction set, the core exposes each extension as an option. A user who only needs the base integer core can leave out floating point, which saves area on silicon.

## 4. The 5-Stage Pipeline

NucleusRV uses the same pipeline structure taught in computer architecture courses, which is part of why it works well as a teaching example.

```
        +------+     +------+     +------+     +------+     +------+
 PC --> |  IF  | --> |  ID  | --> |  EX  | --> | MEM  | --> |  WB  |
        +------+     +------+     +------+     +------+     +------+
        fetch        decode       execute      memory       write
                                                access       back
```

**Stage 1, Instruction Fetch (IF).** Retrieves the next instruction from instruction memory using the current program counter. Because compressed (C extension) instructions can be only 16 bits wide, this stage also contains a Realigner and Compressed Decoder to find instruction boundaries when 16-bit and 32-bit instructions are mixed in the same stream.

**Stage 2, Instruction Decode (ID).** Reads the instruction's opcode, reads the source registers from the register file, builds the immediate value (for instructions like ADDI that carry a constant), and produces the control signals the later stages need. The Hazard Unit lives here, and branch and jump targets are resolved here.

**Stage 3, Execute (EX).** Where the computation happens. The ALU performs arithmetic and logic, and when enabled the multiply/divide unit and the floating-point unit do their work here too. Those units can take more than one clock cycle, so the pipeline can stall on multi-cycle operations. This stage also performs data forwarding, feeding a just-computed result directly to a later stage that needs it instead of waiting for write-back.

**Stage 4, Memory Access (MEM).** Handles reads and writes to data memory for load and store instructions, and manages the multi-step sequencing that atomic (A extension) read-modify-write operations need.

**Stage 5, Write-Back (WB).** Commits the final result into the destination register in the register file, finishing the instruction's path through the pipeline.

## 5. Handling Pipeline Hazards

Running instructions across five overlapping stages creates hazards, and NucleusRV handles each one directly:

- **Data hazards:** solved by forwarding results from the EX and MEM stages back into ID, so a dependent instruction does not have to stall and wait for a value that has already been computed.
- **Load-use hazards:** a special case of a data hazard where a value comes from memory and is not available as early as an ALU result. The pipeline stalls for exactly one cycle to cover this gap.
- **Control hazards:** branches and jumps are resolved as early as the ID stage, which limits the penalty of a taken branch to a single wasted cycle.
- **Atomic hazards:** the read-modify-write sequence that A extension instructions need gets dedicated handling inside the MEM stage so it cannot be corrupted by an overlapping instruction.

The design keeps this manageable by naming every inter-stage pipeline register consistently (for example if_reg_pc, ex_reg_result), so it is clear which stage produced a signal and which stage will consume it.

## 6. Memory Interface

NucleusRV talks to memory through two independent request and response ports rather than one shared bus:

- **Instruction memory port** (imemReq / imemRsp): the fetch stage requests a byte address and gets back a 32-bit instruction word.
- **Data memory port** (dmemReq / dmemRsp): the memory stage uses this for all load and store traffic, including the extra sequencing that atomic operations need.

Keeping instruction and data memory on separate ports (a Harvard-style split) means the core can fetch the next instruction in the same cycle it accesses data memory for a load or store. This is part of what lets a simple 5-stage design stay close to one instruction per cycle.

There is also a small debug output port (io_pin) that exposes retirement data from the write-back stage, which is useful for tracing what the core is doing during simulation.

## 7. Verification Method

Writing a CPU that runs some programs correctly is different from writing one that correctly implements the RISC-V specification in every edge case. NucleusRV verifies itself against the official RISC-V Architectural Test Suite using RISCOF (the RISC-V Compliance Framework), the same class of tooling used to check compliance for production RISC-V implementations.

The verification flow works like this:

1. Test programs from the official riscv-arch-test suite are compiled for each supported extension (I, M, A, F, C).
2. Each program is run both on NucleusRV (through a Verilator-compiled C++ testbench, tb_Top.cpp) and on a reference RISC-V model.
3. RISCOF compares the architectural state (registers and memory) produced by both and flags any mismatch.

This is driven by a single script, run_riscv_arch_tests.py, which runs the whole suite automatically, so compliance can be re-checked after every change to the core.

## 8. Results

The main results from the project are:

- **Compliance:** NucleusRV passes the official RISC-V architectural tests for the extensions it implements (I, M, A, F, C), checked against a reference model through RISCOF. Passing this suite is the standard evidence that a core correctly implements the RISC-V specification.
- **Silicon:** an RV32IMC configuration of the core was fabricated on Google's OpenMPW-6 shuttle using the SkyWater SKY130 open-source PDK. This is a physical tapeout, not a simulation-only result.
- **Simulation:** the Verilator testbench produces a waveform trace (VCD) for each run, which can be opened in a viewer such as GTKWave to inspect signals cycle by cycle.

## 9. Toolchain

Nothing in the design or verification flow needs a proprietary license:

| Purpose | Tool | Open-source |
|---|---|---|
| Hardware design language | Chisel (via Scala and sbt) | Yes |
| RTL simulation | Verilator | Yes |
| C program compilation for the core | RISC-V GNU Toolchain | Yes |
| Compliance testing | RISCOF and riscv-arch-test | Yes |
| Floating-point hardware | berkeley-hardfloat | Yes |
| Silicon fabrication (past tapeout) | SkyWater SKY130 PDK / OpenMPW-6 | Yes |

A student or independent contributor can clone the repository, build the core, run the full compliance suite, and push a configuration through an open silicon flow without a paid EDA license.

## 10. Reproducibility: How to Build and Run

These are the steps to reproduce the core and its tests. They assume a Linux environment with Java, sbt, Verilator, and the RISC-V GNU toolchain installed.

**1. Clone the repository with its submodules** (it depends on berkeley-hardfloat for the floating-point unit):

```bash
git clone --recursive https://github.com/merledu/nucleusrv.git
cd nucleusrv
```

**2. Generate Verilog from the Chisel source** using sbt:

```bash
sbt run
```

**3. Simulate the generated Verilog** with Verilator using the provided C++ testbench. This produces a waveform trace for debugging.

**4. Write or assemble test programs.** Use hand-written RISC-V assembly, or C programs compiled through the RISC-V GNU toolchain, placed in the tools/tests directory.

**5. Run the compliance tests** to confirm the build still matches the RISC-V specification after a change:

```bash
python3 run_riscv_arch_tests.py
```

This loop is what a new contributor, including a student new to RISC-V core design, would go through to start working with the project. For the exact, current commands, follow the README and docs in the repository, since build scripts change over time.

## 11. License

NucleusRV is released under the GNU General Public License v3.0 (GPL-3.0). This notebook documents that project, so it is shared under the same GPL-3.0 license, and a copy of the full license text is included in this submission folder as the LICENSE file.

Open-source licensing is a requirement of the Code-a-Chip grant. Including the license makes clear that the work can be freely studied, reused, and built on by others.

## 12. Summary

NucleusRV is a configurable RV32IMAFC pipeline built and verified with open-source tools, with a real fabrication history on an open-source PDK (SKY130, via OpenMPW-6). It is maintained by a community of contributors and documented at [nucleusrv.readthedocs.io](https://nucleusrv.readthedocs.io/en/latest/).

For anyone who wants to explore open-source processor design, whether to learn pipelining, to experiment with RISC-V extensions, or to push a design toward silicon, NucleusRV is a concrete, working example built end to end on open tooling.

**Repository:** https://github.com/merledu/nucleusrv

**Documentation:** https://nucleusrv.readthedocs.io/en/latest/